# bce-log-loss-real-fake — worked example 3: Discriminator loss from logits via softplus

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `bce-log-loss-real-fake`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

`F.binary_cross_entropy_with_logits` is numerically stable because it never materializes `sigmoid(x)` before taking a log. For a logit `x` and target `1` the loss equals `softplus(-x) = log(1 + exp(-x))`; for target `0` it equals `softplus(x)`. This worked example derives the discriminator loss directly in softplus form so you can see why the fused kernel survives extreme logits.

## Worked solution

**Goal:** compute the discriminator loss from raw logits using `softplus`, and confirm it matches `F.binary_cross_entropy_with_logits`.

1. **BCE-with-logits identity.** For one logit `x`: target 1 gives `-log(sigmoid(x)) = softplus(-x)`; target 0 gives `-log(1 - sigmoid(x)) = softplus(x)`. The `softplus` form avoids ever computing `log(0)`.
2. **Real term.** Real images have target 1, so the per-sample loss is `softplus(-logit_real)`. Mean over the batch.
3. **Fake term.** Fake images have target 0, so the per-sample loss is `softplus(logit_fake)`. Mean over the batch.
4. **Sum** the two means for the combined scalar.
5. **Why this is stable.** At `logit = +50`, `sigmoid(50) = 1.0` in float32, so the naive `-log(1 - sigmoid)` is `-log(0) = inf`. `softplus(50) ≈ 50` exactly — no overflow. We verify equivalence against the fused kernel at moderate logits where both forms are well-defined.

In [ ]:
import torch.nn.functional as F

def disc_loss_logits_softplus(d_logits_real: t.Tensor, d_logits_fake: t.Tensor) -> t.Tensor:
    real_term = F.softplus(-d_logits_real).mean()
    fake_term = F.softplus(d_logits_fake).mean()
    return real_term + fake_term

t.manual_seed(0)
d_logits_real = t.randn(8) * 3.0
d_logits_fake = t.randn(8) * 3.0

mine = disc_loss_logits_softplus(d_logits_real, d_logits_fake)
fused = (F.binary_cross_entropy_with_logits(d_logits_real, t.ones_like(d_logits_real))
         + F.binary_cross_entropy_with_logits(d_logits_fake, t.zeros_like(d_logits_fake)))
print('softplus form:', round(mine.item(), 6))
print('fused with_logits:', round(fused.item(), 6))
print('match:', t.allclose(mine, fused, atol=1e-5))